# Strands Decider — Fine-tuning de Intenções Jurídicas PT-BR

Experimento controlado para treinar o **Strands Decider 2B** com o mesmo dataset jurídico usado no experimento da Laya.

**Protocolo:** 14.806 exemplos de treino → 80% treino / 20% calibração; avaliação congelada em 132 exemplos. O objetivo é medir o modelo, não gerar números de marketing.

O Strands usa Qwen3.5-2B como torso, remove a LM head e usa uma pointer head para pontuar diretamente as opções. O torso recebe LoRA. citeturn3view0

In [ ]:
!nvidia-smi
!git clone -q https://github.com/strands-labs/strands-decider.git /content/strands-decider
%cd /content/strands-decider
!pip install -q -e ".[train,dev]"

In [ ]:
import os, json, math, random, time, hashlib, subprocess, sys
from pathlib import Path
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory/2**30,2))
ROOT=Path('/content/strands-decider')
DATA=ROOT/'data/legal'
DATA.mkdir(parents=True, exist_ok=True)

## 1. Baixar exatamente o dataset jurídico do experimento

O treino e o eval vêm do repositório `suissa/intent-datasets-4finetuning`. O SHA do treino é `cbc70db9...` e o do eval é `9871ef9...`.

In [ ]:
import urllib.request
base='https://raw.githubusercontent.com/suissa/intent-datasets-4finetuning/main/datasets/legal/'
for fn in ['train.jsonl','eval.jsonl','labels.md']:
    urllib.request.urlretrieve(base+fn, DATA/fn)
print((DATA/'train.jsonl').stat().st_size, 'bytes')
print((DATA/'eval.jsonl').stat().st_size, 'bytes')

In [ ]:
LABELS=['CONTRATO','PROCESSO','PETICAO','PRAZO','AUDIENCIA','CONSULTA_JURIDICA','TRABALHISTA','FAMILIA','CONSUMIDOR','EMPRESARIAL','PREVIDENCIARIO','CRIMINAL']
DESCRIPTIONS={
 'CONTRATO':'Questões sobre contratos, cláusulas, assinatura, revisão, rescisão ou obrigações contratuais.',
 'PROCESSO':'Acompanhamento, andamento, situação ou informações sobre um processo judicial.',
 'PETICAO':'Pedido para elaborar, revisar, protocolar ou entender uma petição ou manifestação processual.',
 'PRAZO':'Dúvidas sobre prazos, vencimentos, contagem ou datas de atos jurídicos.',
 'AUDIENCIA':'Questões sobre audiência, comparecimento, preparação, participação ou consequências de uma audiência.',
 'CONSULTA_JURIDICA':'Pedido geral de orientação ou esclarecimento jurídico sem domínio específico claramente identificado.',
 'TRABALHISTA':'Questões de trabalho, emprego, salário, férias, rescisão ou direitos trabalhistas.',
 'FAMILIA':'Questões de família, casamento, divórcio, guarda, alimentos ou relações familiares.',
 'CONSUMIDOR':'Problemas com produtos, serviços, empresas, cobranças, compras ou direitos do consumidor.',
 'EMPRESARIAL':'Questões jurídicas relacionadas a empresas, sociedades, sócios, operações ou negócios.',
 'PREVIDENCIARIO':'Questões sobre INSS, aposentadoria, benefícios previdenciários ou contribuições.',
 'CRIMINAL':'Questões envolvendo crimes, investigação criminal, polícia, acusação, defesa ou processo penal.'
}
print(len(LABELS), LABELS)

## 2. Converter para o formato nativo do Strands

Cada exemplo vira exatamente uma decisão `choice`: estado + pergunta + 12 opções. A ordem das opções fica canônica no arquivo; o próprio Strands embaralha as opções durante o treinamento, que é uma parte importante da arquitetura.

In [ ]:
def load_simple(path):
    rows=[]
    with open(path,encoding='utf-8') as f:
        for line in f:
            if line.strip(): rows.append(json.loads(line))
    return rows

def to_strands(row):
    label=row['label']
    return {
      'kind':'choice',
      'state':f"Mensagem do usuário: {row['text']} [Domínio: legal]",
      'instructions':'Qual categoria jurídica representa melhor a intenção do usuário?',
      'options':[[x, DESCRIPTIONS[x]] for x in LABELS],
      'label':LABELS.index(label),
      'task':'legal_intent_ptbr',
      'weight':1.0,
      'instruction_variants':[
        'Qual é a categoria jurídica que melhor classifica a intenção desta mensagem?',
        'Em qual categoria jurídica esta solicitação deve ser classificada?'
      ]
    }

train=load_simple(DATA/'train.jsonl'); ev=load_simple(DATA/'eval.jsonl')
assert len(train)==14806, len(train)
assert len(ev)==132, len(ev)
random.Random(0).shuffle(train)
cut=11845
train80, calib=train[:cut], train[cut:]
def write_jsonl(path, rows):
    with open(path,'w',encoding='utf-8') as f:
        for r in rows: f.write(json.dumps(to_strands(r),ensure_ascii=False)+'\n')
write_jsonl(DATA/'train_80.jsonl', train80)
write_jsonl(DATA/'calibration_20.jsonl', calib)
 

In [ ]:
from strands_decider.data.format import read_jsonl
print('train:',sum(1 for _ in read_jsonl(DATA/'train_80.jsonl')))
print('calib:',sum(1 for _ in read_jsonl(DATA/'calibration_20.jsonl')))
print('eval:',len(ev))

## 3. Configuração de treino

Para manter o experimento compatível com uma GPU de 12 GB, o notebook usa `micro_batch_size=1`, `grad_accum=32`, `max_length=512` e gradient checkpointing. Em GPUs maiores, aumente o micro-batch mantendo batch efetivo 32.

O torso é `Qwen/Qwen3.5-2B-Base`, LoRA rank 16, pointer head 256 e uma época — alinhado à receita oficial do Strands, adaptando somente tamanho de batch/janela para o hardware. citeturn3view0

In [ ]:
import yaml
CFG={
 'train_files':[str(DATA/'train_80.jsonl')],
 'val_fraction':0.0,
 'base_model':'Qwen/Qwen3.5-2B-Base',
 'num_slots':12,
 'head_type':'pointer',
 'pointer_dim':256,
 'head_init':'random',
 'kl_frozen_weight':0.0,
 'head_dropout':0.05,
 'max_length':512,
 'use_lora':True,
 'lora_r':16,
 'lora_alpha':32,
 'lora_targets':['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj','in_proj_qkv','in_proj_z','in_proj_a','in_proj_b','out_proj'],
 'freeze_torso':False,
 'epochs':1,
 'micro_batch_size':1,
 'grad_accum':32,
 'lr':1e-4,
 'head_lr':1e-3,
 'weight_decay':0.01,
 'warmup_ratio':0.03,
 'max_grad_norm':1.0,
 'gradient_checkpointing':True,
 'shuffle_options':True,
 'group_by_length':True,
 'ordinal_smoothing':0.0,
 'reverse_score_prob':0.5,
 'output_dir':str(ROOT/'checkpoints/legal-ptbr-strands-2b'),
 'seed':0,
 'log_every':20,
 'eval_every':500
}
with open(ROOT/'configs/legal-ptbr.yaml','w') as f: yaml.safe_dump(CFG,f,sort_keys=False)
print(yaml.safe_dump(CFG,sort_keys=False))

## 4. Treinar

**Esta célula executa o treinamento de fato.** Em Colab, não interrompa o runtime durante o epoch se quiser o checkpoint final.

In [ ]:
!python -m strands_decider.cli train --config /content/strands-decider/configs/legal-ptbr.yaml

## 5. Calibrar com os 20% reservados

A calibração é feita somente nos 2.961 exemplos separados do treino. Ela não altera o argmax; ajusta a temperatura das probabilidades para que confiança tenha significado operacional.

In [ ]:
from strands_decider.infer import load_engine, EngineConfig
from strands_decider.modeling import StrandsDeciderModel
from strands_decider.evaluate import collect_logits, fit_temperature_by_kind
from strands_decider.data.format import load_examples

ckpt=str(ROOT/'checkpoints/legal-ptbr-strands-2b')
calib_ex=load_examples([str(DATA/'calibration_20.jsonl')])
engine_cal=load_engine(ckpt,device='cuda')
model=engine_cal.model
logits, labels, slots, _=collect_logits(model, calib_ex, device='cuda', batch_size=8, max_length=512)
temps=fit_temperature_by_kind(logits,labels,slots,calib_ex,min_examples=200,objective='ece')
print('temperaturas:',temps)

In [ ]:
from strands_decider.modeling import StrandsDeciderConfig, config_path, checkpoint_dir
cfg=StrandsDeciderConfig.from_json(config_path(checkpoint_dir(ckpt)))
cfg.temperature_by_kind=temps
with open(config_path(checkpoint_dir(ckpt)),'w',encoding='utf-8') as f: f.write(cfg.to_json())
print('Temperaturas salvas no checkpoint:',cfg.temperature_by_kind)

## 6. Avaliação congelada — os mesmos 132 casos da Laya

In [ ]:
from strands_decider.evaluate import predictions_from_logits
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
import numpy as np

eval_json=[]
with open(DATA/'eval.jsonl',encoding='utf-8') as f:
    eval_json=[json.loads(x) for x in f if x.strip()]
eval_ex=[to_strands(x) for x in eval_json]
from strands_decider.data.format import Example
eval_ex=[Example.from_dict(x) for x in eval_ex]
elogits, elabels, eslots, _=collect_logits(model,eval_ex,device='cuda',batch_size=8,max_length=512)
preds=predictions_from_logits(elogits,elabels,eslots,eval_ex,temperature=temps)
y=[p.label for p in preds]; yp=[p.pred for p in preds]; conf=[p.confidence for p in preds]
acc=accuracy_score(y,yp); mf1=f1_score(y,yp,average='macro')
print(f'Accuracy: {acc:.6f}')
print(f'Macro-F1: {mf1:.6f}')
print(classification_report(y,yp,labels=list(range(12)),target_names=LABELS,zero_division=0))

In [ ]:
def brier_multiclass(preds):
    vals=[]
    for p in preds:
        s=sum((q-(1.0 if i==p.label else 0.0))**2 for i,q in enumerate(p.probs))
        vals.append(s)
    return float(np.mean(vals))

def ece(preds,bins=10):
    c=np.array([p.confidence for p in preds]); ok=np.array([p.correct for p in preds],dtype=float)
    total=len(preds); out=0.0
    for lo,hi in zip(np.linspace(0,1,bins+1)[:-1],np.linspace(0,1,bins+1)[1:]):
        m=(c>=lo)&((c<hi) if hi<1 else (c<=hi))
        if m.any(): out += m.mean()*abs(ok[m].mean()-c[m].mean())
    return float(out)
print('Brier:',brier_multiclass(preds))
print('ECE:',ece(preds))
print('Mean confidence:',float(np.mean(conf)))
print('Mean confidence correct:',float(np.mean([p.confidence for p in preds if p.correct])))
print('Mean confidence wrong:',float(np.mean([p.confidence for p in preds if not p.correct])) if any(not p.correct for p in preds) else 0.0)

In [ ]:
errors=[]
for i,p in enumerate(preds):
    if not p.correct:
        errors.append({'i':i,'text':eval_json[i]['text'],'gold':LABELS[p.label],'pred':LABELS[p.pred],'confidence':p.confidence,'probs':dict(zip(LABELS,p.probs))})
for e in errors: print(e['gold'],'->',e['pred'],f"conf={e['confidence']:.3f}",e['text'])
print('errors:',len(errors),'/',len(preds))

## 7. Latência local

Mede somente inferência local do checkpoint, sem rede. O benchmark anterior da Laya usou p50/p95; aqui usamos o mesmo conceito.

In [ ]:
engine=load_engine(ckpt,device='cuda')
sample_states=[x['text'] for x in eval_json]
criteria={k:DESCRIPTIONS[k] for k in LABELS}
q={'intent':{'type':'choice','instructions':'Qual categoria jurídica representa melhor a intenção do usuário?','criteria':criteria}}
times=[]
for s in sample_states:
    torch.cuda.synchronize(); t0=time.perf_counter()
    engine.ask(f'Mensagem do usuário: {s} [Domínio: legal]', {
      'intent': __import__('strands_decider.schema',fromlist=['ChoiceQuestion']).ChoiceQuestion(instructions=q['intent']['instructions'],criteria=criteria)
    })
    torch.cuda.synchronize(); times.append((time.perf_counter()-t0)*1000)
print('p50 ms:',float(np.percentile(times,50)))
print('p95 ms:',float(np.percentile(times,95)))

## 8. Salvar resultados


In [ ]:
report={
 'model':'Strands Decider legal PT-BR',
 'base_model':'Qwen/Qwen3.5-2B-Base',
 'train_examples':11845,
 'calibration_examples':2961,
 'eval_examples':132,
 'accuracy':float(acc),
 'macro_f1':float(mf1),
 'brier':brier_multiclass(preds),
 'ece':ece(preds),
 'mean_confidence':float(np.mean(conf)),
 'mean_confidence_correct':float(np.mean([p.confidence for p in preds if p.correct])),
 'mean_confidence_wrong':float(np.mean([p.confidence for p in preds if not p.correct])) if any(not p.correct for p in preds) else None,
 'latency_p50_ms':float(np.percentile(times,50)),
 'latency_p95_ms':float(np.percentile(times,95)),
  'temperature':temps,
 'errors':errors
}
Path('/content/strands_legal_results.json').write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding='utf-8')
print(json.dumps(report,ensure_ascii=False,indent=2))

## Interpretação

Não use a acurácia isoladamente. Para este caso, a ordem de importância é: **accuracy/Macro-F1 → erros nos pares de confusão → ECE/Brier → confiança dos erros → latência**. Um modelo que acerta 97% mas afirma 100% nos erros é pior para automação do que um modelo ligeiramente menos preciso e bem calibrado.

Depois deste notebook, compare o resultado com o checkpoint Laya já medido e com uma avaliação Tev1 feita sobre estes mesmos 132 exemplos.